# 🎙️ Urdu / Hinglish voice test — your own cloned voice (free GPU)

Compares two free, **commercial-OK** voice-cloning models on the same bilingual lines:

| Model | Urdu | License |
|---|---|---|
| **OmniVoice** (k2-fsa) | ✅ real Urdu (`ur`, ~211 h training data) | Apache-2.0 |
| **Chatterbox Multilingual** (Resemble AI) | Hindi (`hi`) — Hindustani in your voice | MIT |

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`. Then run the cells **top to bottom** (▶ on each, or `Runtime → Run all`).

**Tip:** cloning copies the *accent of the recording you upload*. Your English sample works, but for the most natural Urdu, also try uploading a 10-second clip of you speaking **Urdu**.

## 1 · Install OmniVoice (~2 min)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install omnivoice soundfile
print("✅ OmniVoice installed")

## 2 · Upload your voice (any format: m4a, mp3, wav, ogg — 10 s to 60 s, quiet room)

In [ ]:
from google.colab import files
import subprocess, os
up = files.upload()
src = list(up.keys())[0]
print("uploaded:", src)
# Full clean reference (24 kHz mono) + an 8-second clip (OmniVoice works best with 3-10 s).
subprocess.run(["ffmpeg","-y","-v","error","-i",src,"-af","highpass=f=70,loudnorm=I=-16:TP=-1.5","-ar","24000","-ac","1","ref_full.wav"],check=True)
subprocess.run(["ffmpeg","-y","-v","error","-ss","0.5","-t","8","-i","ref_full.wav","ref_clip.wav"],check=True)
from IPython.display import Audio, display
print("8-second reference clip:")
display(Audio("ref_clip.wav"))

## 3 · Test lines
Edit these freely — keep the same idea in each style so you can compare fairly.

In [ ]:
TESTS = {
  "urdu_script":   ("ur", "آج ہم ایک ایسی ٹیکنالوجی کی بات کریں گے جو سب کچھ بدل دے گی۔ کیا آپ تیار ہیں؟"),
  "urdu_english":  ("ur", "آج ہم AI کے بارے میں بات کریں گے، اور یہ technology سب کچھ change کر دے گی۔"),
  "roman_urdu":    ("ur", "Aaj hum baat karenge ek aisi technology ki jo sab kuch change kar degi. Kya aap ready hain?"),
  "hindi":         ("hi", "आज हम एक ऐसी टेक्नोलॉजी की बात करेंगे जो सब कुछ बदल देगी। क्या आप तैयार हैं?"),
  "hindi_english": ("hi", "आज हम AI के बारे में बात करेंगे, और ये technology सब कुछ change कर देगी।"),
  "english":       ("en", "Today we're talking about a technology that will change everything. Are you ready?"),
}
os.makedirs("out", exist_ok=True)
print(len(TESTS), "test lines ready")

## 4 · OmniVoice — your voice

In [ ]:
import torch, inspect, soundfile as sf
from omnivoice import OmniVoice
from IPython.display import Audio, display
model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
params = inspect.signature(model.generate).parameters
LANG_ARG = next((p for p in ("language", "language_id", "lang") if p in params), None)
print("generate() language argument:", LANG_ARG or "(none — language is inferred from the text)")
# Transcribe the reference once and reuse it (faster than re-transcribing for every line).
prompt = model.create_voice_clone_prompt(ref_audio="ref_clip.wav") if hasattr(model, "create_voice_clone_prompt") else None
for name, (lang, text) in TESTS.items():
    kw = {"text": text}
    kw.update({"voice_clone_prompt": prompt} if prompt is not None else {"ref_audio": "ref_clip.wav"})
    if LANG_ARG: kw[LANG_ARG] = lang
    try:
        audio = model.generate(**kw)
        path = f"out/omnivoice_{name}.wav"
        sf.write(path, audio[0], 24000)
        print(f"▶ OmniVoice · {name}:  {text}")
        display(Audio(path))
    except Exception as e:
        print(f"✗ OmniVoice · {name}: {e}")
del model; torch.cuda.empty_cache()

## 5 · Chatterbox Multilingual — your voice
Installs a second model. **If Colab says "Restart session", click it, then run cell 2's upload again only if `ref_clip.wav` is gone, and continue from this cell.**

In [ ]:
!pip -q install chatterbox-tts
print("✅ Chatterbox installed")

In [ ]:
import torch, torchaudio as ta, os
from IPython.display import Audio, display
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
cb = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
os.makedirs("out", exist_ok=True)
CB_TESTS = {k: v for k, v in TESTS.items() if v[0] in ("hi", "en")}  # Chatterbox has Hindi + English, not Urdu
for name, (lang, text) in CB_TESTS.items():
    try:
        wav = cb.generate(text, language_id=lang, audio_prompt_path="ref_full.wav")
        path = f"out/chatterbox_{name}.wav"
        ta.save(path, wav, cb.sr)
        print(f"▶ Chatterbox · {name}:  {text}")
        display(Audio(path))
    except Exception as e:
        print(f"✗ Chatterbox · {name}: {e}")

## 6 · Download every clip

In [ ]:
!cd out && zip -q -r ../voice_test_clips.zip . && cd ..
from google.colab import files
files.download("voice_test_clips.zip")

## Which one won?
Tell Claude which clips sounded most like **you** and most natural in **Urdu / mixed**, e.g. *"OmniVoice urdu_english is best"* — the winner gets wired into **Studio — Make Reel** as a language option.